# 2026-06-13 Day 1: Orientation, Shape Contract, Embeddings, Loss

Goal: understand the repo and the decoder-only LM path from token IDs to logits and cross entropy.


## How To Use This Reading Notebook

Use this before touching implementation for the day. The goal is not to memorize the paper. The goal is to extract the model of the system you need for today's code and notes.

Workflow:

1. Read only the assigned sections.
2. Fill the mini-lecture answer in your own words.
3. Open the repo files listed in the roadmap.
4. Run the listed checks or record why they skip.
5. Write a short exit ticket before moving on.


## Assigned Reading

| Source | Exact Target | Why It Matters Today |
| --- | --- | --- |
| [Attention Is All You Need](https://arxiv.org/abs/1706.03762) | Sections 3.1 and 3.4 | Establishes encoder/decoder framing, embeddings, and the final linear-softmax vocabulary projection. |
| [nanoGPT](https://github.com/karpathy/nanoGPT) | `model.py` skim only | Gives practical file organization and tensor naming, especially `B, T, C`. |


## Reading Summary

### Attention Is All You Need, Section 3.1

The original Transformer is presented as an encoder-decoder sequence transduction model. For this repo, the important transfer is the decoder-side idea: a stack of blocks maps token representations to next-token predictions while preserving sequence length. The original paper includes encoder-decoder attention, but the project uses a decoder-only language model, so the central idea is the self-attention stack with a causal mask.

Key takeaway for implementation: every block receives a sequence of token vectors and returns a sequence of token vectors. The model does not collapse time before the LM head.

### Attention Is All You Need, Section 3.4

Tokens are converted to vectors with learned embeddings, and final hidden states are projected back to vocabulary logits. The paper also adds positional information because attention alone does not know token order. In this repo, the current trainable model uses learned token and position embeddings.

Key takeaway for implementation: embedding lookup changes `(B, T)` token IDs into `(B, T, C)` hidden states, and the LM head changes `(B, T, C)` into `(B, T, V)` logits.

### nanoGPT `model.py` Skim

Read nanoGPT for naming and organization, not as something to copy line-for-line. Notice that the code uses compact tensor names: `B` for batch, `T` for sequence length, and `C` for channels or hidden width. The model is organized around config, attention, MLP, block, and GPT wrapper.

Key takeaway for implementation: this repo already follows a similar separation with `ModelConfig`, attention/block/model classes, and tests for shape behavior.


## Diagram Anchor

![Decoder shape ladder](../../assets/figures/decoder_shape_ladder.svg)

What to notice: targets remain integer IDs with shape `(B, T)`. They are not one-hot vectors in the training API. Cross entropy handles the vocabulary comparison from logits.

![Shifted targets](../../assets/figures/embedding_shifted_targets.svg)

What to notice: the input and target sequences are nearly the same text, offset by one token.


## Repo Connection

Open these files after reading:

| File | What To Find |
| --- | --- |
| `src/moe_transformer_lab/config.py` | `ModelConfig` and fields that define `V`, `T`, `C`, layers, heads, and FFN width. |
| `src/moe_transformer_lab/manual/layers.py` | `Embedding`, `Linear`, and `SoftmaxCrossEntropy`. |
| `src/moe_transformer_lab/trainable/model.py` | `token_embedding`, `position_embedding`, `blocks`, `norm`, and `lm_head`. |
| `tests/test_swappable_ffn.py` | Forward shape check for dense and MoE variants. |


## Mini-Lecture Answer Seed

A decoder-only language model starts with `idx` shaped `(B, T)`. Embedding lookup maps each integer token to a vector, producing `(B, T, C)`. Position information is added with the same shape. Every transformer block preserves `(B, T, C)`. The final LM head projects the last dimension from hidden width `C` to vocabulary size `V`, producing logits `(B, T, V)`. For cross entropy, logits are flattened to `(B*T, V)` and targets are flattened to `(B*T)`.


## Active Recall

1. Why is the target tensor `(B, T)` instead of `(B, T, V)`?
2. Which operation first introduces `C`?
3. Which operation first introduces `V`?
4. Why can dense and MoE models share the same loss?

## Exit Ticket

Before ending Day 1, write:

- the full shape path from token IDs to loss;
- embedding and LM-head parameter formulas;
- one attention-shape question to resolve on Day 2.
